# Prediksi Harga Properti dari Teks Penjualan
**HoloMine Task 2, Hology 9.0 | Tim Nanahoshi**

Soal ini meminta kami menebak `listPrice` sebuah listing properti hanya dari teks deskripsinya. Tidak ada kolom luas, jumlah kamar, atau lokasi. Semua informasi harus dicari sendiri dari kalimat yang ditulis agen. Metrik penilaiannya MAE pada harga asli, jadi kesalahan pada rumah mahal terasa jauh lebih berat daripada pada rumah murah.

Alur notebook ini:

1. Melihat data dulu: distribusi harga, panjang teks, kata yang khas, angka di dalam teks, ekor harga, listing kembar, cek kebocoran, dan perbandingan sebaran train dengan test.
2. Menyiapkan fitur: angka dari regex, target dalam skala log, pembagian fold, dan bobot koreksi pergeseran distribusi.
3. Model level satu (TF-IDF dan embedding kalimat), lalu digabung dengan LightGBM.
4. Ablasi tiap tahap, analisis error, dan file submission.

Dijalankan dari atas ke bawah di Kaggle dengan GPU T4 dan internet menyala. Perkiraan sekitar 1 sampai 1,5 jam. Bagian Qwen paling lambat dan bisa dimatikan lewat `PAKAI_QWEN = False` kalau waktunya mepet. Hasil tiap tahap disimpan di folder cache, jadi kalau sesi terputus tinggal jalankan ulang dan tahap yang sudah selesai dilewati.

In [ ]:
!pip install -q -U "transformers>=4.51" sentencepiece lightgbm

## 1. Persiapan

In [ ]:
import os, re, gc, glob, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.sparse import hstack
from scipy.stats import spearmanr
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer, ENGLISH_STOP_WORDS
from sklearn.linear_model import Ridge, RidgeCV, LogisticRegression
from sklearn.svm import LinearSVR, SVR
from sklearn.decomposition import TruncatedSVD
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import roc_auc_score
import lightgbm as lgb
from joblib import Parallel, delayed

warnings.filterwarnings("ignore")
SEED, NF = 42, 5
SUBSAMPLE = 0
PAKAI_FINETUNE = True       # tahap 5.4, matikan kalau waktu mepet
FT_EPOCH = 4                # percobaan sebelumnya berhenti di 3 dan belum konvergen
FT_MAXLEN = 320             # p95 token 420; dipotong demi kecepatan
FT_BATAS_MENIT = 105        # jangan mulai fold baru setelah menit ini
BOBOT_ADVERSARIAL = True
CACHE = "/kaggle/working/cache"
OUT = "/kaggle/working"
os.makedirs(CACHE, exist_ok=True)

plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.25,
                     "axes.spines.top": False, "axes.spines.right": False})
t_mulai = time.time()
def lapor(*a):
    print(f"[{(time.time() - t_mulai) / 60:5.1f} mnt]", *a, flush=True)

## 2. Data

In [ ]:
folder = os.path.dirname(glob.glob("/kaggle/input/**/train.csv", recursive=True)[0])
train = pd.read_csv(f"{folder}/train.csv")
test = pd.read_csv(f"{folder}/test.csv")
contoh = pd.read_csv(f"{folder}/sample_submission.csv")
if SUBSAMPLE:
    train = train.sample(SUBSAMPLE, random_state=0).reset_index(drop=True)
    test = test.head(max(60, SUBSAMPLE // 4)).reset_index(drop=True)
train["text"] = train["text"].fillna("")
test["text"] = test["text"].fillna("")

y = train["listPrice"].values.astype(float)
ly = np.log(np.clip(y, 1, None))
semua_teks = list(train["text"]) + list(test["text"])

print("train:", train.shape, "| test:", test.shape)
print("nilai kosong di train:", int(train.isna().sum().sum()), "| teks duplikat persis:", int(train["text"].duplicated().sum()))
print("panjang teks (karakter), median:", int(train["text"].str.len().median()), "| maks:", int(train["text"].str.len().max()))
train.head(3)

## 3. Melihat data (EDA)

### 3.1 Distribusi harga

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 3.8))
ax[0].hist(y[y < 3e6], bins=80, color="#4c72b0")
ax[0].set_title("Harga, dipotong sampai 3 juta")
ax[0].set_xlabel("listPrice")
ax[1].hist(np.log10(np.clip(y, 1, None)), bins=80, color="#55a868")
ax[1].set_title("Harga dalam skala log10")
ax[1].set_xlabel("log10(listPrice)")
urut = np.sort(y)
ax[2].plot(urut, np.arange(1, len(urut) + 1) / len(urut), color="#c44e52")
ax[2].set_xscale("log")
ax[2].set_title("Distribusi kumulatif")
ax[2].set_xlabel("listPrice (skala log)")
plt.tight_layout()
plt.show()

pd.Series(y).describe(percentiles=[.01, .05, .25, .5, .75, .95, .99]).round(0).to_frame("listPrice")

Harga sangat miring ke kanan. Sebagian besar listing ada di kisaran ratusan ribu, tetapi ada yang mencapai puluhan juta. Di skala log bentuknya jauh lebih simetris. Ini alasan pertama kami memodelkan `log(listPrice)` dan bukan harga mentahnya.

### 3.2 Panjang teks

In [ ]:
panjang = train["text"].str.len().values
plt.figure(figsize=(7, 4.2))
plt.hexbin(panjang, np.log10(np.clip(y, 1, None)), gridsize=45, cmap="viridis", mincnt=1, bins="log")
plt.colorbar(label="jumlah listing (log)")
plt.xlabel("panjang teks (karakter)")
plt.ylabel("log10(listPrice)")
plt.title("Panjang teks dan harga")
plt.show()
print("korelasi Spearman panjang teks dengan harga:", round(spearmanr(panjang, y)[0], 3))

Ada hubungan positif yang lumayan. Deskripsi rumah mahal cenderung lebih panjang karena banyak fasilitas yang dijelaskan. Panjang teks kami ikutkan sebagai fitur. Nanti di bagian 3.7 fakta ini muncul lagi dan ternyata penting untuk alasan yang berbeda.

### 3.3 Kata yang membedakan listing murah dan mahal

Untuk setiap kata yang muncul di minimal 40 listing, kami hitung median harga listing yang memuat kata itu. Kata dengan median terendah dan tertinggi ditampilkan di bawah.

In [ ]:
henti = list(ENGLISH_STOP_WORDS) + ["redacted", "entity"]
cv = CountVectorizer(binary=True, min_df=40, stop_words=henti, token_pattern=r"(?u)\b[a-zA-Z]{3,}\b")
M = cv.fit_transform(train["text"]).tocsc()
kosakata = np.array(cv.get_feature_names_out())
median_kata = np.array([np.median(ly[M.indices[M.indptr[j]:M.indptr[j + 1]]]) for j in range(M.shape[1])])
o = np.argsort(median_kata)

fig, ax = plt.subplots(1, 2, figsize=(13, 4.6))
murah, mahal = o[:15], o[::-1][:15]
ax[0].barh(kosakata[murah][::-1], np.exp(median_kata[murah][::-1]), color="#55a868")
ax[0].set_title("Kata dengan median harga terendah")
ax[1].barh(kosakata[mahal][::-1], np.exp(median_kata[mahal][::-1]), color="#c44e52")
ax[1].set_title("Kata dengan median harga tertinggi")
for a in ax:
    a.set_xlabel("median listPrice")
plt.tight_layout()
plt.show()

Sisi murah diisi nama kota dan kata yang berkaitan dengan tanah kosong atau rumah yang perlu diperbaiki, misalnya `survey`, `camping`, `rehab`. Sisi mahal berisi material dan gaya hunian mewah seperti `limestone`, `gunite`, `mansion`, juga nama kawasan. Jadi teks membawa dua hal yang jelas berhubungan dengan harga, yaitu jenis properti dan lokasi. Model bahasa yang sudah banyak membaca teks semacam ini kemungkinan menangkapnya lebih baik daripada hitungan kata biasa, dan itu yang akan kami uji.

### 3.4 Angka yang tersembunyi di dalam teks

Jumlah kamar, luas, dan tahun bangun tidak diberikan sebagai kolom, tetapi kadang disebut di dalam kalimat. Kami tulis fungsi regex untuk mengambilnya, dan fungsi ini dipakai lagi di bagian pengolahan data.

In [ ]:
ANGKA_KATA = {"one": 1, "two": 2, "three": 3, "four": 4, "five": 5, "six": 6, "seven": 7, "eight": 8, "nine": 9, "ten": 10}

def ke_angka(s):
    try:
        return float(s.replace(",", ""))
    except ValueError:
        return np.nan

def cari(pola, s):
    m = re.search(pola, s, re.I)
    return ke_angka(m.group(1)) if m else np.nan

def ekstrak(teks):
    k = teks.lower()
    k2 = re.sub(r"\b(one|two|three|four|five|six|seven|eight|nine|ten)\b", lambda m: str(ANGKA_KATA[m.group(1)]), k)
    d = {
        "kamar_tidur": cari(r"(\d+)[\s-]*(?:bed|br\b|bd\b|bedroom)", k2),
        "kamar_mandi": cari(r"(\d+\.?\d*)[\s-]*(?:bath|ba\b)", k2),
        "sqft": cari(r"([\d,]{3,7})\s*(?:\+/-\s*)?(?:sq\.?\s*ft|sqft|square f|sf\b)", k),
        "acre": cari(r"(\d[\d,]*\.?\d*)\s*(?:\+/-\s*)?acre", k),
        "lot_sqft": cari(r"lot[^.]{0,30}?([\d,]{4,8})\s*(?:sq|sf)", k),
        "garasi": cari(r"(\d)[\s-]*(?:car)\s*garage", k2),
        "lantai": cari(r"(\d)[\s-]*(?:story|stories)", k2),
        "unit": cari(r"(\d+)[\s-]*(?:unit|plex)", k2),
    }
    tahun = [int(t) for t in re.findall(r"\b(1[6-9]\d\d|20[0-2]\d)\b", k) if 1800 <= int(t) <= 2026]
    m = re.search(r"built (?:in )?(1[6-9]\d\d|20[0-2]\d)", k)
    d["tahun"] = float(m.group(1)) if m else (float(tahun[0]) if tahun else np.nan)
    dolar = re.findall(r"\$\s?([\d,]{4,12})", teks)
    d["dolar_maks"] = max(ke_angka(x) for x in dolar) if dolar else np.nan
    d["dolar_n"] = len(dolar)
    d["panjang"] = len(teks)
    d["kata"] = len(teks.split())
    d["redacted"] = teks.count("[Redacted")
    d["seru"] = teks.count("!")
    d["digit"] = sum(c.isdigit() for c in teks)
    d["rasio_kapital"] = sum(c.isupper() for c in teks) / max(1, len(teks))
    return d

F_tr = pd.DataFrame([ekstrak(t) for t in train["text"]])
F_te = pd.DataFrame([ekstrak(t) for t in test["text"]])

utama = ["kamar_tidur", "kamar_mandi", "sqft", "acre", "tahun", "garasi", "lantai", "unit"]
liputan = F_tr[utama].notna().mean().sort_values()
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].barh(liputan.index, liputan.values * 100, color="#4c72b0")
ax[0].set_xlabel("% listing yang menyebutkan")
ax[0].set_title("Seberapa sering angka ini muncul di teks")
ada_mandi = F_tr["kamar_mandi"].notna()
tabel = pd.DataFrame({"mandi": F_tr.loc[ada_mandi, "kamar_mandi"].round().clip(upper=8), "y": y[ada_mandi]}).groupby("mandi")["y"].median()
ax[1].bar(tabel.index.astype(int), tabel.values, color="#c44e52")
ax[1].set_xlabel("jumlah kamar mandi")
ax[1].set_ylabel("median listPrice")
ax[1].set_title("Median harga menurut jumlah kamar mandi")
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
for a, kol, nama in [(ax[0], "acre", "luas lahan (acre)"), (ax[1], "sqft", "luas bangunan (sqft)")]:
    ok = F_tr[kol].notna() & (F_tr[kol] > 0)
    a.hexbin(np.log10(F_tr.loc[ok, kol]), np.log10(np.clip(y[ok], 1, None)), gridsize=40, cmap="magma", mincnt=1, bins="log")
    a.set_xlabel(f"log10({nama})")
    a.set_ylabel("log10(listPrice)")
plt.tight_layout()
plt.show()

Angka-angka ini hanya muncul di sebagian listing. Jumlah kamar tidur paling sering disebut, sekitar separuh listing, sedangkan luas bangunan, luas lahan, dan tahun bangun jauh lebih jarang. Kalau ada, angkanya berguna: median harga naik seiring jumlah kamar mandi dan luas lahan punya hubungan positif dengan harga. Luas bangunan tidak menunjukkan pola naik yang rapi. Dugaan kami, unit kecil di kota mahal bercampur dengan rumah besar di kota yang murah. Karena sering kosong, angka ini kami jadikan fitur tambahan untuk model pohon, yang bisa menangani nilai kosong, dan bukan satu-satunya sumber informasi.

### 3.5 Ekor harga dan kenapa MAE sulit

In [ ]:
selisih = np.sort(np.abs(y - np.median(y)))[::-1]
bagian = np.arange(1, len(selisih) + 1) / len(selisih)
plt.figure(figsize=(6.5, 4))
plt.plot(bagian * 100, np.cumsum(selisih) / selisih.sum() * 100, color="#c44e52")
plt.xlabel("% listing (diurutkan dari selisih terbesar)")
plt.ylabel("% total selisih absolut")
plt.title("Sedikit listing menyumbang sebagian besar selisih")
plt.show()
for f in [0.01, 0.05, 0.10, 0.25]:
    print(f"{int(f * 100):>3}% listing teratas menyumbang {selisih[:int(len(selisih) * f)].sum() / selisih.sum():.0%} dari total selisih terhadap median")

Lima persen listing termahal menyumbang hampir setengah dari total selisih terhadap median. Artinya nilai MAE nanti akan sangat ditentukan oleh seberapa baik model menebak rumah mahal. Model yang bagus di rentang menengah tetapi meleset jauh pada rumah puluhan juta tetap akan punya MAE besar.

Satu catatan yang baru kami sadari setelah analisis error di bagian 6: kenyataan ini sering disalahartikan. Ekor memang menguasai MAE, tetapi itu tidak otomatis berarti prediksi di segmen mahal perlu dinaikkan. Pembahasannya ada di bagian 6.2.

### 3.6 Listing yang mirip dan pengecekan id

Kami cek dua hal. Pertama, apakah ada listing yang hampir sama antara train dan test. Kedua, apakah nomor id punya hubungan dengan harga, karena kalau ya itu tanda data bocor dan tidak boleh dipakai.

In [ ]:
tf = TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)
A = tf.fit_transform(train["text"])
B = tf.transform(test["text"])

def kemiripan_terdekat(P, Q, sama=False):
    hasil = np.zeros(P.shape[0])
    for s in range(0, P.shape[0], 1000):
        S = (P[s:s + 1000] @ Q.T).toarray()
        if sama:
            S[np.arange(S.shape[0]), np.arange(s, s + S.shape[0])] = -1
        hasil[s:s + 1000] = S.max(1)
    return hasil

sim_train = kemiripan_terdekat(A, A, sama=True)
sim_test = kemiripan_terdekat(B, A)
plt.figure(figsize=(7, 4))
plt.hist(sim_train, bins=50, alpha=0.6, density=True, label="train ke train")
plt.hist(sim_test, bins=50, alpha=0.6, density=True, label="test ke train")
plt.yscale("log")
plt.xlabel("kemiripan cosine dengan listing terdekat")
plt.legend()
plt.show()
print(f"kemiripan di atas 0,9: train {np.mean(sim_train > 0.9):.1%}, test {np.mean(sim_test > 0.9):.1%}")

nomor = train["id"].str.extract(r"(\d+)")[0].astype(int)
print("korelasi Spearman id dengan harga:", round(spearmanr(nomor, y)[0], 3))
urut_id = np.argsort(nomor.values)
ly_urut = ly[urut_id]
print("selisih |log harga| id bersebelahan:", round(float(np.median(np.abs(np.diff(ly_urut)))), 4))
acak = np.random.default_rng(0).permutation(len(ly))
print("selisih |log harga| pasangan acak  :", round(float(np.median(np.abs(ly[acak[:-1]] - ly[acak[1:]]))), 4))

Listing yang hampir kembar jumlahnya kecil, hanya beberapa persen, dan porsinya di test sedikit lebih rendah daripada di train. Jadi model tidak bisa mengandalkan pencocokan teks saja.

Nomor id tidak berhubungan dengan harga. Kami juga cek apakah id yang berdekatan punya harga yang mirip, karena itu pola kebocoran yang umum kalau data diurutkan menurut wilayah saat dikumpulkan. Selisihnya sama saja dengan pasangan acak, jadi id benar-benar tidak membawa informasi dan tidak dipakai.

### 3.7 Apakah datanya dibangkitkan mesin

Kalau teksnya hasil template, biasanya ada pola yang bisa dibalik dan itu jalan pintas besar. Ada tim lain di papan peringkat dengan skor jauh di bawah yang masuk akal, jadi kami cek kemungkinan itu sebelum lanjut memodelkan.

In [ ]:
gabung_teks = pd.Series(semua_teks)
print(f"teks unik   : {gabung_teks.nunique()} dari {len(gabung_teks)} ({gabung_teks.nunique()/len(gabung_teks):.4f})")

kal = []
for t in train["text"].head(4000):
    kal += [s.strip() for s in re.split(r"(?<=[.!?])\s+", t) if len(s.strip()) > 25]
vc = pd.Series(kal).value_counts()
print(f"kalimat unik: {len(vc)} dari {len(kal)} ({len(vc)/len(kal):.3f})")

pola_caps = r"\b[A-Z]{4,}\b"
print(f"listing yang memuat kata KAPITAL SEMUA: {train['text'].str.contains(pola_caps).mean():.3f}")
print()
print("kalimat yang paling sering berulang:")
for s, c in vc.head(4).items():
    print(f"  {c:3d}x  {s[:70]}")

Teksnya unik seratus persen dan kalimatnya unik sekitar 99 persen. Tidak ada pool template. Yang berulang cuma basa basi agen seperti ajakan menjadwalkan kunjungan. Sekitar 18 persen listing memuat kata berhuruf kapital semua, ciri tulisan agen sungguhan, bukan keluaran mesin.

Kesimpulannya datanya asli dan tidak ada generator yang bisa dibalik. Satu-satunya jalan adalah memodelkan makna teksnya.

### 3.8 Apakah train dan test datang dari sebaran yang sama

Ini pemeriksaan yang hampir kami lewatkan dan ternyata paling berguna di seluruh notebook.

Caranya dengan adversarial validation: gabung train dan test, beri label 0 dan 1, lalu latih classifier untuk membedakan keduanya. Kalau AUC-nya 0,5 berarti sebarannya identik dan CV bisa dipercaya apa adanya. Kalau jauh di atas 0,5 berarti ada pergeseran, dan angka CV akan menipu.

In [ ]:
berkas_adv = f"{CACHE}/prob_adv.npy"
if os.path.exists(berkas_adv):
    p_adv = np.load(berkas_adv)
else:
    lab_adv = np.r_[np.zeros(len(train)), np.ones(len(test))]
    X_adv = TfidfVectorizer(ngram_range=(1, 2), min_df=3, max_features=100000, sublinear_tf=True).fit_transform(gabung_teks)
    p_adv = cross_val_predict(LogisticRegression(max_iter=1000), X_adv, lab_adv, cv=3, method="predict_proba")[:, 1]
    np.save(berkas_adv, p_adv)
    lapor(f"adversarial validation selesai, AUC {roc_auc_score(lab_adv, p_adv):.4f}")

lab_adv = np.r_[np.zeros(len(train)), np.ones(len(test))]
print(f"AUC train lawan test: {roc_auc_score(lab_adv, p_adv):.4f}   (0,5 berarti sebaran identik)")
print()
pola_red = r"\[Redacted"
print(f"median panjang teks   train {train['text'].str.len().median():.0f}  |  test {test['text'].str.len().median():.0f}")
print(f"memuat [Redacted]     train {train['text'].str.contains(pola_red).mean():.3f}  |  test {test['text'].str.contains(pola_red).mean():.3f}")

plt.figure(figsize=(7.5, 4))
b = np.linspace(0, 3000, 60)
plt.hist(train["text"].str.len().clip(upper=3000), bins=b, density=True, alpha=0.65, label="train", color="#4c72b0")
plt.hist(test["text"].str.len().clip(upper=3000), bins=b, density=True, alpha=0.55, label="test", color="#c44e52")
plt.xlabel("panjang teks (karakter)")
plt.ylabel("kerapatan")
plt.title("Teks di test lebih panjang daripada di train")
plt.legend()
plt.show()

In [ ]:
bobot = np.clip(p_adv[:len(train)] / np.clip(1 - p_adv[:len(train)], 1e-6, None), 0, 8)
bobot = bobot / bobot.mean()

print(f"bobot: p10 {np.quantile(bobot, .1):.2f} | median {np.median(bobot):.2f} | p90 {np.quantile(bobot, .9):.2f} | maks {bobot.max():.2f}")
print(f"rata-rata harga tanpa bobot   : {y.mean():,.0f}")
print(f"rata-rata harga setelah dibobot: {np.average(y, weights=bobot):,.0f}")

AUC-nya sekitar 0,59. Bukan angka yang bisa diabaikan. Teks di test lebih panjang dan lebih sering mengandung `[Redacted Entity]`. Dari bagian 3.2 kita sudah tahu teks panjang berarti properti mahal, jadi test condong ke properti yang lebih mahal daripada train. Ekornya lebih berat, dan MAE-nya akan lebih besar.

Ini menjelaskan sebagian jarak yang sebelumnya membingungkan kami: submission kami dengan pipeline serupa mendapat skor publik sekitar 358 ribu padahal MAE CV-nya 291 ribu. Sebelumnya kami hanya menduga subset publik kebetulan lebih berat.

Dari probabilitas classifier tadi kami hitung bobot importance sampling, lalu dipakai untuk dua hal. Pertama, menimbang ulang metrik validasi supaya mendekati sebaran test. Kedua, sebagai sample weight saat melatih stacker. Yang kedua belum tentu membantu, jadi nanti di bagian 5.5 kami uji dua-duanya dan laporkan apa adanya.

**Yang kami simpulkan dari EDA**

- Harga miring ke kanan dan MAE didominasi rumah mahal, jadi target dimodelkan dalam skala log dengan loss berbasis median (L1 dan Huber).
- Informasi harga ada di jenis properti, lokasi, dan gaya bahasa, bukan hanya di angka. Representasi teks yang memahami makna kalimat layak dicoba di samping TF-IDF.
- Angka terstruktur hanya ada di sebagian teks, jadi dipakai sebagai fitur pelengkap.
- Datanya asli, tidak ada template atau kebocoran lewat id.
- Test bergeser ke arah yang lebih mahal, jadi selain CV biasa kami laporkan CV ala-test.
- Karena ekor berat, fold harus memastikan tiap fold punya campuran harga yang mirip.

## 4. Pengolahan data

### 4.1 Fitur angka dan pembagian fold

Fitur regex sudah dibuat di bagian 3.4 (`F_tr` dan `F_te`). Untuk validasi kami memakai 5 fold yang distratifikasi berdasarkan 20 kelompok kuantil harga, supaya rumah mahal yang jarang muncul terbagi rata di semua fold.

In [ ]:
kelompok_harga = pd.qcut(ly, 20, labels=False, duplicates="drop")
fold = np.zeros(len(train), dtype=int)
for k, (_, va) in enumerate(StratifiedKFold(NF, shuffle=True, random_state=SEED).split(train, kelompok_harga)):
    fold[va] = k

pd.DataFrame({"jumlah": np.bincount(fold),
              "median": [np.median(y[fold == k]) for k in range(NF)],
              "rata-rata": [y[fold == k].mean() for k in range(NF)],
              "p99": [np.percentile(y[fold == k], 99) for k in range(NF)]}).round(0)

### 4.2 Target, metrik, dan lantai derau

Model dilatih pada `log(listPrice)`. Kalau model memprediksi median bersyarat dalam skala log, hasilnya setelah `exp` juga median bersyarat dalam skala harga, dan median adalah penduga terbaik untuk MAE. Karena itu loss yang dipakai L1 dan Huber, bukan kuadrat.

Kami pakai dua metrik. `mae` adalah MAE biasa pada seluruh data train. `mae_test` adalah MAE yang ditimbang dengan bobot dari bagian 3.8, yaitu perkiraan MAE kalau data validasinya punya sebaran seperti test.

Satu hal lagi yang perlu diketahui sebelum membandingkan model: seberapa kecil selisih yang masih berarti. Kami ukur dengan bootstrap pada subset seukuran leaderboard publik.

In [ ]:
def dari_log(z):
    return np.exp(np.clip(z, 0, np.log(3e8)))

def mae(p):
    return float(np.mean(np.abs(y - p)))

def mae_test(p):
    return float(np.average(np.abs(y - p), weights=bobot))

def simpan(nama, oof, tes):
    np.save(f"{CACHE}/oof_{nama}.npy", oof)
    np.save(f"{CACHE}/tes_{nama}.npy", tes)

def ada(nama):
    return os.path.exists(f"{CACHE}/oof_{nama}.npy")

N_PUBLIK = max(50, int(len(test) * 0.30))

def ambang_derau(err, n_ulang=2000):
    """Simpangan baku MAE kalau hanya dihitung pada subset seukuran leaderboard publik."""
    rng = np.random.default_rng(0)
    contoh_mae = [err[rng.choice(len(err), N_PUBLIK, replace=False)].mean() for _ in range(n_ulang)]
    return float(np.std(contoh_mae))

dasar = np.full(len(y), np.median(y))
print(f"MAE kalau semua ditebak median : {mae(dasar):,.0f}")
print(f"MAE ala-test untuk tebakan itu : {mae_test(dasar):,.0f}")

sd_dasar = ambang_derau(np.abs(y - dasar))
print()
print(f"Lantai derau pada tebakan konstan, subset {N_PUBLIK} baris (30 persen test):")
print(f"  simpangan baku antar subset: {sd_dasar:,.0f}")
print("  angka ini dihitung ulang memakai error model sebenarnya di bagian 5.3.")

### 4.3 Model level satu dari TF-IDF

Beberapa model teks klasik. Semuanya dilatih per fold, dan prediksinya pada data validasi fold itu (out-of-fold) menjadi fitur untuk model tingkat dua. Prediksi untuk test adalah rata-rata dari lima model fold.

- Ridge pada TF-IDF kata (1-2 gram) digabung karakter (3-5 gram), dan Ridge pada kata saja.
- kNN berbasis cosine pada TF-IDF kata: harga tetangga terdekat, median tetangga, dan seberapa mirip tetangganya. Bagian 3.6 menunjukkan ada listing yang hampir kembar, jadi skor kemiripan ikut dimasukkan supaya model tingkat dua tahu kapan harus mempercayainya.
- SVD 32 komponen sebagai ringkasan topik.
- LinearSVR dan Ridge pada teks yang semua digitnya diganti `0`, supaya angka tidak menjadi ribuan token berbeda.
- Ridge pada kata berhuruf kapital saja, sebagai pendekatan kasar untuk nama tempat.

In [ ]:
NAMA_TFIDF = (["ridge_cw", "ridge_w", "knn_w", "knn_med", "knn_1", "knn_s1", "knn_s3", "knn_m3", "knn_sd"]
              + [f"svd{i}" for i in range(32)] + ["svr_w", "ridge_prop", "ridge_norm"])

def huruf_kapital(t):
    return " ".join(re.findall(r"(?<![.!?]\s)(?<!^)\b[A-Z][a-z]{2,}\b", t))

def angka_nol(t):
    return re.sub(r"\d", "0", t.lower())

def tetangga(Wb, Wa, ya, k=10):
    hasil = []
    for s in range(0, Wb.shape[0], 1000):
        S = (Wb[s:s + 1000] @ Wa.T).toarray()
        kk = min(k, S.shape[1] - 1)
        idx = np.argpartition(-S, kk, axis=1)[:, :kk]
        sim = np.take_along_axis(S, idx, 1)
        o = np.argsort(-sim, axis=1)
        idx, sim = np.take_along_axis(idx, o, 1), np.take_along_axis(sim, o, 1)
        p = ya[idx]
        w = np.maximum(sim, 1e-6) ** 4
        hasil.append(np.c_[(p * w).sum(1) / w.sum(1), np.median(p, 1), p[:, 0], sim[:, 0], sim[:, :3].mean(1), p[:, :3].mean(1), p.std(1)])
    return np.vstack(hasil)

def fitur_tfidf(txt_a, ya, txt_b):
    kata = TfidfVectorizer(ngram_range=(1, 2), min_df=3, max_features=200000, sublinear_tf=True)
    karakter = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=5, max_features=200000, sublinear_tf=True)
    Wa, Wb = kata.fit_transform(txt_a), kata.transform(txt_b)
    Ca, Cb = karakter.fit_transform(txt_a), karakter.transform(txt_b)
    ridge_cw = Ridge(alpha=3.0).fit(hstack([Wa, Ca]).tocsr(), ya).predict(hstack([Wb, Cb]).tocsr())
    ridge_w = Ridge(alpha=1.0).fit(Wa, ya).predict(Wb)
    knn = tetangga(Wb, Wa, ya)
    svd = TruncatedSVD(32, random_state=0).fit(Wa).transform(Wb)
    norm = TfidfVectorizer(ngram_range=(1, 3), min_df=3, max_features=300000, sublinear_tf=True)
    Da, Db = norm.fit_transform(txt_a.map(angka_nol)), norm.transform(txt_b.map(angka_nol))
    svr = LinearSVR(C=0.3, epsilon=0.0, loss="epsilon_insensitive", max_iter=5000, random_state=0).fit(Da, ya).predict(Db)
    ridge_norm = Ridge(alpha=2.0).fit(Da, ya).predict(Db)
    kap = TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)
    ridge_prop = Ridge(alpha=1.0).fit(kap.fit_transform(txt_a.map(huruf_kapital)), ya).predict(kap.transform(txt_b.map(huruf_kapital)))
    return pd.DataFrame(np.c_[ridge_cw, ridge_w, knn, svd, svr, ridge_prop, ridge_norm], columns=NAMA_TFIDF)

if not ada("ridge_cw"):
    oof = pd.DataFrame(0.0, index=range(len(train)), columns=NAMA_TFIDF)
    tes = np.zeros((len(test), len(NAMA_TFIDF)))
    for k in range(NF):
        a, b = np.where(fold != k)[0], np.where(fold == k)[0]
        F = fitur_tfidf(train["text"].iloc[a], ly[a], pd.concat([train["text"].iloc[b], test["text"]], ignore_index=True))
        oof.iloc[b] = F.iloc[:len(b)].values
        tes += F.iloc[len(b):].values / NF
        lapor(f"tfidf fold {k} selesai")
    for j, n in enumerate(NAMA_TFIDF):
        simpan(n, oof[n].values, tes[:, j])
else:
    lapor("tfidf: sudah ada di cache")

ringkas_tfidf = {}
for n in ["ridge_cw", "ridge_w", "knn_w", "knn_med", "svr_w", "ridge_norm", "ridge_prop"]:
    p = dari_log(np.load(f"{CACHE}/oof_{n}.npy"))
    ringkas_tfidf[n] = {"MAE CV": mae(p), "MAE ala-test": mae_test(p)}
pd.DataFrame(ringkas_tfidf).T.sort_values("MAE ala-test").round(0)

Perhatikan jarak antara kedua kolom. Selisihnya konsisten di semua model dan itulah ongkos pergeseran distribusi yang ditemukan di bagian 3.8.

## 5. Pengembangan model

### 5.1 Model tingkat dua: LightGBM

Semua prediksi model level satu, ditambah fitur regex, dimasukkan ke LightGBM. Kami melatih dua versi, satu dengan loss L1 dan satu dengan Huber, lalu merata-ratakan prediksinya di skala log. Jumlah iterasi dipilih dengan early stopping pada potongan 10 persen dari data latih di dalam tiap fold, jadi fold validasi tidak pernah dipakai untuk memilih iterasi. Prediksi test dari model akhir dilatih pada seluruh train dengan tiga seed.

Fungsi `stack_cv` menerima argumen `w`, yaitu sample weight. Kalau `None`, stacker dilatih tanpa bobot. Ini dipakai di bagian 5.5 untuk menguji apakah bobot adversarial benar-benar membantu.

In [ ]:
DASAR = dict(learning_rate=0.03, num_leaves=31, min_data_in_leaf=20, feature_fraction=0.7,
             bagging_fraction=0.8, bagging_freq=1, lambda_l2=1.0, verbose=-1, num_threads=-1)
OBJ = {"l1": dict(DASAR, objective="l1"), "huber": dict(DASAR, objective="huber", alpha=1.0)}

def stack_cv(X, w=None):
    oof, iters = {}, {}
    for nama, p in OBJ.items():
        o, it = np.zeros(len(X)), []
        for k in range(NF):
            a, b = np.where(fold != k)[0], np.where(fold == k)[0]
            dalam = np.random.default_rng(k).choice(a, size=len(a) // 10, replace=False)
            latih = np.setdiff1d(a, dalam)
            d1 = lgb.Dataset(X.iloc[latih], ly[latih], weight=None if w is None else w[latih])
            d2 = lgb.Dataset(X.iloc[dalam], ly[dalam], weight=None if w is None else w[dalam])
            m = lgb.train(p, d1, 3000, valid_sets=[d2], callbacks=[lgb.early_stopping(100, verbose=False)])
            o[b] = m.predict(X.iloc[b], num_iteration=m.best_iteration)
            it.append(m.best_iteration)
        oof[nama], iters[nama] = o, max(50, int(np.mean(it) * 1.1))
    return (oof["l1"] + oof["huber"]) / 2, oof, iters

def stack_akhir(X, Xt, iters, w=None):
    p = np.zeros(len(Xt))
    for nama, param in OBJ.items():
        for s in range(3):
            m = lgb.train(dict(param, seed=s, bagging_seed=s, feature_fraction_seed=s),
                          lgb.Dataset(X, ly, weight=w), iters[nama])
            p += m.predict(Xt) / (3 * len(OBJ))
    return p

KELOMPOK = {"tfidf": NAMA_TFIDF}
HASIL = {}

def rakit(kelompok):
    nama = [n for g in kelompok for n in KELOMPOK[g]]
    Xa = pd.concat([F_tr, pd.DataFrame({n: np.load(f"{CACHE}/oof_{n}.npy") for n in nama})], axis=1)
    Xb = pd.concat([F_te, pd.DataFrame({n: np.load(f"{CACHE}/tes_{n}.npy") for n in nama})], axis=1)
    return Xa, Xb

def evaluasi(label, kelompok, w="auto"):
    if w == "auto":
        w = bobot if BOBOT_ADVERSARIAL else None
    Xa, Xb = rakit(kelompok)
    campur, per_obj, iters = stack_cv(Xa, w)
    HASIL[label] = dict(cv=mae(dari_log(campur)), cv_test=mae_test(dari_log(campur)),
                        l1=mae(dari_log(per_obj["l1"])), huber=mae(dari_log(per_obj["huber"])),
                        oof=campur, tes=stack_akhir(Xa, Xb, iters, w), iters=iters, kelompok=kelompok)
    h = HASIL[label]
    lapor(f"{label}: CV {h['cv']:,.0f} | ala-test {h['cv_test']:,.0f}")

evaluasi("TF-IDF + regex", ["tfidf"])

### 5.2 Embedding kalimat

TF-IDF hanya menghitung kata. Embedding dari model bahasa yang sudah dilatih mengubah seluruh teks menjadi satu vektor yang menyimpan makna, termasuk pengetahuan umum seperti kawasan mana yang mahal. Kami memakai tiga model tanpa fine-tuning, jadi cukup satu kali lewat model untuk seluruh teks:

- `gte-modernbert-base`, encoder berbasis ModernBERT.
- `bge-large-en-v1.5` dan `e5-large-v2`, dua encoder BERT ukuran large.

Ketiganya encoder-only dari keluarga BERT, bukan model generatif, dan dipakai beku sebagai pengekstrak fitur. Kami sempat mencoba `Qwen3-Embedding-0.6B` yang kualitasnya lebih tinggi, tetapi model itu dibangun dari backbone LLM, jadi kami tidak memakainya dan sakelarnya dimatikan.

Di atas tiap embedding kami latih tiga model kecil per fold: Ridge, kNN, dan SVR dengan kernel RBF. SVR biasanya yang terbaik tetapi paling lambat, jadi fold-nya dijalankan paralel.

In [ ]:
import torch
import torch.nn.functional as Fn
from transformers import AutoTokenizer, AutoModel
from joblib import Parallel, delayed

DEV = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEV)

CFG = {
    "e5_l":  dict(nama="intfloat/e5-large-v2",            pool="mean", awalan="query: ", panjang=512, bs=32),
    "gte":   dict(nama="Alibaba-NLP/gte-modernbert-base", pool="cls",  awalan="",        panjang=512, bs=64),
    "e5_b":  dict(nama="intfloat/e5-base-v2",             pool="mean", awalan="query: ", panjang=512, bs=64),
    "bge_l": dict(nama="BAAI/bge-large-en-v1.5",          pool="cls",  awalan="",        panjang=512, bs=32),
    "bge":   dict(nama="BAAI/bge-base-en-v1.5",           pool="cls",  awalan="",        panjang=512, bs=64),
}

def encode(cfg, teks, setengah=True):
    tok = AutoTokenizer.from_pretrained(cfg["nama"])
    model = AutoModel.from_pretrained(cfg["nama"])
    model = model.half() if (setengah and DEV == "cuda") else model.float()
    model = model.to(DEV).eval()
    pad = tok.pad_token_id if tok.pad_token_id is not None else 0
    ids = tok([cfg["awalan"] + t for t in teks], truncation=True, max_length=cfg["panjang"])["input_ids"]
    panjang = np.array([len(i) for i in ids])
    urut = np.argsort(panjang)
    hasil, rusak, bs, t0 = None, 0, cfg["bs"], time.time()
    with torch.no_grad():
        for s in range(0, len(urut), bs):
            b = urut[s:s + bs]
            x = torch.full((len(b), panjang[b].max()), pad, dtype=torch.long)
            m = torch.zeros(x.shape, dtype=torch.long)
            for j, i in enumerate(b):
                x[j, :panjang[i]] = torch.tensor(ids[i])
                m[j, :panjang[i]] = 1
            x, m = x.to(DEV), m.to(DEV)
            h = model(input_ids=x, attention_mask=m).last_hidden_state.float()
            if cfg["pool"] == "cls":
                v = h[:, 0]
            else:
                mm = m.unsqueeze(-1).float()
                v = (h * mm).sum(1) / mm.sum(1).clamp(min=1)
            v = Fn.normalize(v, dim=-1)
            rusak += int((~torch.isfinite(v)).any(1).sum())
            v = torch.nan_to_num(v)
            if hasil is None:
                hasil = np.zeros((len(teks), v.shape[1]), np.float32)
            hasil[b] = v.cpu().numpy()
            if (s // bs) % 80 == 0 and s:
                print(f"   {s}/{len(urut)}, sisa sekitar {(time.time()-t0)/s*(len(urut)-s)/60:.1f} menit", flush=True)
    del model
    gc.collect()
    if DEV == "cuda":
        torch.cuda.empty_cache()
    return hasil, rusak

def model_kecil(k, Etr, Ete, fold, ly):
    a, b = np.where(fold != k)[0], np.where(fold == k)[0]
    mu, sd = ly[a].mean(), ly[a].std()
    r = RidgeCV(alphas=np.logspace(-2, 1, 7)).fit(Etr[a], ly[a])
    def knn(Q):
        S = Q @ Etr[a].T
        kk = min(20, S.shape[1] - 1)
        top = np.argpartition(-S, kk, axis=1)[:, :kk]
        sim = np.take_along_axis(S, top, 1)
        w = np.exp((sim - sim.max(1, keepdims=True)) / 0.02)
        return (w * ly[a][top]).sum(1) / w.sum(1)
    s = SVR(C=3.0, epsilon=0.05, cache_size=1500).fit(Etr[a], (ly[a] - mu) / sd)
    return dict(b=b, ridge=(r.predict(Etr[b]), r.predict(Ete)), knn=(knn(Etr[b]), knn(Ete)),
                svr=(s.predict(Etr[b]) * sd + mu, s.predict(Ete) * sd + mu))

def tahap_embedding(kode):
    nama = [f"{kode}_ridge", f"{kode}_knn", f"{kode}_svr"]
    KELOMPOK[kode] = nama
    if all(ada(n) for n in nama):
        lapor(f"{kode}: sudah ada di cache")
        return
    berkas = f"{CACHE}/emb_{kode}.npy"
    if os.path.exists(berkas):
        E = np.load(berkas)
    else:
        lapor(f"{kode}: menghitung embedding ({CFG[kode]['nama']})")
        E, rusak = encode(CFG[kode], semua_teks, setengah=True)
        if rusak:
            lapor(f"{kode}: {rusak} vektor tidak valid di fp16, diulang di fp32")
            E, _ = encode(CFG[kode], semua_teks, setengah=False)
        np.save(berkas, E)
    lapor(f"{kode}: dimensi {E.shape[1]}, melatih model kecil per fold")
    Etr, Ete = E[:len(train)], E[len(train):]
    hasil = Parallel(n_jobs=max(1, min(NF, os.cpu_count() or 1)))(
        delayed(model_kecil)(k, Etr, Ete, fold, ly) for k in range(NF))
    for m in ["ridge", "knn", "svr"]:
        oof, tes = np.zeros(len(train)), np.zeros(len(test))
        for h in hasil:
            oof[h["b"]] = h[m][0]
            tes += h[m][1] / NF
        simpan(f"{kode}_{m}", oof, tes)
    skor = {m: mae(dari_log(np.load(f"{CACHE}/oof_{kode}_{m}.npy"))) for m in ["ridge", "knn", "svr"]}
    lapor(f"{kode}: " + ", ".join(f"{m} {v:,.0f}" for m, v in skor.items()))

def tulis_submission(label=""):
    """Tulis submission dari tahap terbaik sejauh ini, dipanggil tiap tahap selesai."""
    t = min(HASIL, key=lambda k: HASIL[k]["cv_test"])
    p = dari_log(HASIL[t]["tes"])
    sub = contoh[["id"]].merge(pd.DataFrame({"id": test["id"].values, "listPrice": np.round(p, 2)}),
                               on="id", how="left")
    sub["listPrice"] = sub["listPrice"].fillna(float(np.median(p)))
    sub.to_csv(f"{OUT}/submission.csv", index=False)
    lapor(f"submission ditulis dari tahap '{t}' {label} (ala-test {HASIL[t]['cv_test']:,.0f})")

In [ ]:
URUT = ["e5_l", "gte", "e5_b", "bge_l", "bge"]
dipakai = ["tfidf"]
for kode in URUT:
    try:
        tahap_embedding(kode)
        dipakai.append(kode)
        evaluasi(f"+ {kode}", list(dipakai))
        tulis_submission(f"setelah {kode}")
    except Exception as e:
        lapor(f"{kode} GAGAL: {type(e).__name__}: {str(e)[:160]}")
        gc.collect()

In [ ]:
# encoder dijalankan di sel sebelumnya dalam satu perulangan

In [ ]:
print("urutan tahap:", list(HASIL.keys()))

### 5.4 Fine-tune encoder untuk tugas ini

Semua model di bagian 5.2 adalah encoder beku. Vektornya dilatih orang lain untuk mengukur
kemiripan makna secara umum, bukan untuk menebak harga. Dari tabel ablasi kelihatan
encoder kedua hampir tidak menambah apa-apa, karena informasi yang dibawanya tumpang
tindih dengan encoder pertama.

Jadi di sini kami coba sesuatu yang berbeda jenisnya: melatih ulang seluruh bobot
`ModernBERT-base` langsung pada target harga. Modelnya encoder-only seperti BERT, bukan
model generatif. Bedanya dengan bagian 5.2, model ini belajar sendiri fitur apa yang
relevan untuk harga, bukan memakai fitur kemiripan umum.

Beberapa keputusan teknis:

- Empat epoch. Di percobaan kami sebelumnya latihan dihentikan di epoch tiga, dan MAE
  validasinya masih turun 7 sampai 19 ribu di setiap epoch, artinya model belum selesai
  belajar.
- Panjang token dipotong di 384. Persentil 95 panjang token sekitar 420, jadi yang
  terpotong hanya sedikit, sementara waktunya jauh lebih hemat.
- Prediksi diambil dari epoch terakhir, bukan epoch terbaik menurut fold validasi. Kalau
  kami memilih epoch terbaik memakai fold validasi, skor OOF-nya jadi optimistis.
- Layer-wise learning rate decay 0,9, jadi lapisan bawah berubah lebih pelan daripada
  lapisan atas. Dropout dimatikan, praktik umum untuk regresi.
- Tiap fold disimpan ke cache begitu selesai. Kalau sesi terputus, jalankan ulang dan
  fold yang sudah jadi tidak diulang.

Ada pengaman waktu. Kalau sampai menit ke `FT_BATAS_MENIT` masih ada fold yang belum
mulai, tahap ini dilewati dan notebook memakai model terbaik dari bagian 5.3. Submission
dari tahap sebelumnya sudah ditulis, jadi tidak ada yang hilang.

In [ ]:
FT_SIAP = False
try:
    import torch
    import torch.nn as nn
    from transformers import AutoConfig, AutoModel, AutoTokenizer, get_cosine_schedule_with_warmup

    FT_NAMA = "answerdotai/ModernBERT-base"
    FT_DIR = f"{CACHE}/ft_mbert"
    os.makedirs(FT_DIR, exist_ok=True)

    def bikin_regresor(path):
        cfg = AutoConfig.from_pretrained(path)
        for k in ("hidden_dropout_prob", "attention_probs_dropout_prob", "attention_dropout",
                  "embedding_dropout", "mlp_dropout", "classifier_dropout", "pooler_dropout"):
            if hasattr(cfg, k):
                setattr(cfg, k, 0.0)

        class Reg(nn.Module):
            def __init__(s):
                super().__init__()
                # .float() penting: transformers baru bisa memuat bobot fp16 dan GradScaler
                # menolaknya. Bobot master harus fp32, fp16 hanya lewat autocast.
                s.bb = AutoModel.from_pretrained(path, config=cfg).float()
                s.head = nn.Linear(cfg.hidden_size, 1)

            def forward(s, ids, mask):
                h = s.bb(input_ids=ids, attention_mask=mask).last_hidden_state
                m = mask.unsqueeze(-1).to(h.dtype)
                return s.head((h * m).sum(1) / m.sum(1).clamp(min=1)).squeeze(-1)
        return Reg(), cfg

    def grup_parameter(model, n_layer, lr, head_lr, llrd, wd=0.01):
        g = {}
        for nama, p in model.named_parameters():
            if nama.startswith("head"):
                lr_p = head_lr
            else:
                m = re.search(r"layers?\.(\d+)\.", nama)
                dalam = 0 if ("embeddings" in nama and not m) else (int(m.group(1)) + 1 if m else n_layer)
                lr_p = lr * (llrd ** (n_layer - dalam))
            nd = p.ndim == 1 or nama.endswith(".bias")
            g.setdefault((lr_p, 0.0 if nd else wd), []).append(p)
        return [dict(params=v, lr=k[0], weight_decay=k[1]) for k, v in g.items()]

    def batch_urut(panjang, bs, acak, rng):
        idx = np.arange(len(panjang))
        if not acak:
            idx = idx[np.argsort(panjang)]
            return [idx[i:i + bs] for i in range(0, len(idx), bs)]
        rng.shuffle(idx)
        keluar = []
        for s in range(0, len(idx), bs * 32):
            blok = idx[s:s + bs * 32]
            blok = blok[np.argsort(panjang[blok])]
            keluar += [blok[i:i + bs] for i in range(0, len(blok), bs)]
        return [keluar[i] for i in rng.permutation(len(keluar))]

    def susun(enc, ids, pad):
        L = max(len(enc[i]) for i in ids)
        x = torch.full((len(ids), L), pad, dtype=torch.long)
        m = torch.zeros((len(ids), L), dtype=torch.long)
        for j, i in enumerate(ids):
            x[j, :len(enc[i])] = torch.tensor(enc[i])
            m[j, :len(enc[i])] = 1
        return x, m

    def latih_fold(k, enc, panjang, te_enc, te_panjang, pad, dev):
        torch.manual_seed(SEED + k)
        rng = np.random.default_rng(SEED + k)
        a, b = np.where(fold != k)[0], np.where(fold == k)[0]
        model, mcfg = bikin_regresor(FT_NAMA)
        model.to(dev)
        mu, sd = ly[a].mean(), ly[a].std()
        z = (ly - mu) / sd
        opt = torch.optim.AdamW(grup_parameter(model, mcfg.num_hidden_layers, 4e-5, 5e-4, 0.9))
        per_epoch = len(batch_urut(panjang[a], 16, True, np.random.default_rng(0)))
        total = per_epoch * FT_EPOCH
        sched = get_cosine_schedule_with_warmup(opt, int(0.06 * total), total)
        amp = dev.type == "cuda"
        scaler = torch.amp.GradScaler("cuda", enabled=amp)
        rugi_fn = nn.SmoothL1Loss(beta=0.1)

        def tebak(semua, e, p):
            model.eval()
            keluar = np.zeros(len(semua))
            with torch.no_grad(), torch.autocast(device_type=dev.type, dtype=torch.float16, enabled=amp):
                for bb in batch_urut(p[semua], 64, False, None):
                    sel = semua[bb]
                    x, m = susun(e, sel, pad)
                    keluar[bb] = model(x.to(dev), m.to(dev)).float().cpu().numpy()
            return keluar

        t0, langkah = time.time(), 0
        for ep in range(FT_EPOCH):
            model.train()
            for bb in batch_urut(panjang[a], 16, True, rng):
                sel = a[bb]
                x, m = susun(enc, sel, pad)
                y_t = torch.tensor(z[sel], dtype=torch.float32, device=dev)
                with torch.autocast(device_type=dev.type, dtype=torch.float16, enabled=amp):
                    pred = model(x.to(dev), m.to(dev))
                rugi = rugi_fn(pred.float(), y_t)
                opt.zero_grad(set_to_none=True)
                scaler.scale(rugi).backward()
                scaler.unscale_(opt)
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(opt)
                scaler.update()
                sched.step()
                langkah += 1
                if langkah % 400 == 0:
                    laju = langkah / (time.time() - t0)
                    print(f"    fold {k} langkah {langkah}/{total}, sisa {(total-langkah)/laju/60:.1f} menit", flush=True)
            pv = tebak(b, enc, panjang) * sd + mu
            lapor(f"  fold {k} epoch {ep+1}/{FT_EPOCH}: val MAE {mae(dari_log(pv)):,.0f}")
        pt = tebak(np.arange(len(te_panjang)), te_enc, te_panjang) * sd + mu
        del model, opt
        gc.collect()
        if dev.type == "cuda":
            torch.cuda.empty_cache()
        return b, pv, pt

    def jalankan_finetune():
        dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        tok = AutoTokenizer.from_pretrained(FT_NAMA)
        enc = tok(list(train["text"]), truncation=True, max_length=FT_MAXLEN)["input_ids"]
        te_enc = tok(list(test["text"]), truncation=True, max_length=FT_MAXLEN)["input_ids"]
        panjang = np.array([len(e) for e in enc])
        te_panjang = np.array([len(e) for e in te_enc])
        pad = tok.pad_token_id if tok.pad_token_id is not None else 0
        lapor(f"token: median {int(np.median(panjang))}, p95 {int(np.percentile(panjang,95))}, "
              f"terpotong {np.mean(panjang >= FT_MAXLEN):.1%}, device {dev}")

        for k in range(NF):
            berkas = f"{FT_DIR}/fold{k}.npz"
            if os.path.exists(berkas):
                lapor(f"  fold {k} sudah ada di cache")
                continue
            lewat = (time.time() - t_mulai) / 60
            if lewat > FT_BATAS_MENIT:
                lapor(f"  berhenti di fold {k}, sudah menit ke {lewat:.0f} (batas {FT_BATAS_MENIT})")
                return False
            b, pv, pt = latih_fold(k, enc, panjang, te_enc, te_panjang, pad, dev)
            np.savez(berkas, b=b, pv=pv, pt=pt)
            lapor(f"  fold {k} disimpan")

        berkas = [f"{FT_DIR}/fold{k}.npz" for k in range(NF)]
        if not all(os.path.exists(f) for f in berkas):
            lapor("  fold belum lengkap, tahap ini dilewati")
            return False
        oof, tes = np.zeros(len(train)), np.zeros(len(test))
        for f in berkas:
            d = np.load(f)
            oof[d["b"]] = d["pv"]
            tes += d["pt"] / NF
        simpan("ft_mbert", oof, tes)
        lapor(f"  ModernBERT fine-tune: CV {mae(dari_log(oof)):,.0f} | ala-test {mae_test(dari_log(oof)):,.0f}")
        return True

    FT_SIAP = False
    if PAKAI_FINETUNE:
        try:
            FT_SIAP = jalankan_finetune()
        except Exception as e:
            lapor(f"fine-tune gagal: {type(e).__name__}: {str(e)[:200]}")
            FT_SIAP = False
    else:
        lapor("fine-tune dimatikan lewat PAKAI_FINETUNE")
except Exception as e:
    lapor(f'tahap fine-tune dilewati: {type(e).__name__}: {str(e)[:200]}')
    FT_SIAP = False

In [ ]:
if FT_SIAP:
    KELOMPOK["ft"] = ["ft_mbert"]
    evaluasi("+ fine-tune", dipakai + ["ft"])
    tulis_submission("setelah fine-tune")
else:
    print("tahap fine-tune tidak dipakai, model terbaik tetap dari tahap sebelumnya")

Kalau baris `+ fine-tune` muncul di tabel ablasi bagian 5.3 dengan penurunan yang lebih
besar daripada lantai derau, berarti melatih encoder untuk tugas ini memang memberi
informasi yang tidak dimiliki encoder beku. Kalau penurunannya lebih kecil dari lantai
derau, kesimpulannya sebaliknya, dan itu juga jawaban yang sah.

### 5.5 Ablasi tiap tahap

Tabel ini menunjukkan seberapa besar tiap tambahan membantu. Semua angka adalah MAE out-of-fold di data train, bukan skor leaderboard.

In [ ]:
ringkas = pd.DataFrame({k: {"L1": v["l1"], "Huber": v["huber"], "CV": v["cv"], "CV ala-test": v["cv_test"]}
                        for k, v in HASIL.items()}).T.round(0)
ringkas["Turun (ala-test)"] = (-ringkas["CV ala-test"].diff()).round(0)
display(ringkas)

fig, ax = plt.subplots(figsize=(7.5, 3.8))
x = np.arange(len(ringkas))
ax.bar(x - 0.2, ringkas["CV"], width=0.38, color="#4c72b0", label="CV biasa")
ax.bar(x + 0.2, ringkas["CV ala-test"], width=0.38, color="#c44e52", label="CV ala-test")
ax.set_xticks(x)
ax.set_xticklabels(ringkas.index, rotation=15)
ax.set_ylim(min(ringkas["CV"]) * 0.9, max(ringkas["CV ala-test"]) * 1.03)
ax.set_ylabel("MAE")
ax.set_title("MAE tiap tahap")
ax.legend()
plt.tight_layout()
plt.show()

TERBAIK = min(HASIL, key=lambda k: HASIL[k]["cv_test"])
AMBANG = ambang_derau(np.abs(y - dari_log(HASIL[TERBAIK]["oof"])))
print("Tahap dengan MAE ala-test terendah:", TERBAIK)
print(f"Lantai derau dari error model ini pada subset {N_PUBLIK} baris: {AMBANG:,.0f}")
print("Selisih antar tahap yang lebih kecil dari itu belum tentu nyata.")

Kolom terakhir menunjukkan tambahan mana yang benar-benar membantu. Bacalah bersama angka lantai derau yang dicetak di bawah tabel: selisih yang lebih kecil dari itu jangan ditafsirkan sebagai perbaikan pasti.

Di percobaan kami sebelumnya dengan pipeline serupa, menambahkan embedding kalimat ke TF-IDF menurunkan MAE CV sekitar 12 ribu, jauh lebih besar daripada pengaturan detail apa pun di model tingkat dua. Angka untuk run ini ada di tabel di atas.

Kami juga sempat mencoba fine-tuning ModernBERT penuh di percobaan terpisah. Hasil tunggalnya 347 ribu, tidak lebih baik dari embedding beku dengan SVR yang 346 ribu, dan sumbangannya pada model gabungan hanya sekitar dua ribu poin dengan waktu latih lebih dari satu jam. Karena itu tidak kami masukkan ke notebook ini.

### 5.4 Fitur yang paling dipakai

In [ ]:
h = HASIL[TERBAIK]
Xa, Xb = rakit(h["kelompok"])
w_pakai = bobot if BOBOT_ADVERSARIAL else None
model = lgb.train(dict(OBJ["l1"], seed=0), lgb.Dataset(Xa, ly, weight=w_pakai), h["iters"]["l1"])
penting = pd.Series(model.feature_importance("gain"), index=Xa.columns).sort_values(ascending=False).head(15)
plt.figure(figsize=(7, 4.5))
plt.barh(penting.index[::-1], penting.values[::-1] / penting.sum() * 100, color="#55a868")
plt.xlabel("% dari total gain")
plt.title("15 fitur terpenting di LightGBM (L1)")
plt.tight_layout()
plt.show()

### 5.5 Apakah bobot adversarial benar-benar membantu

Bagian 3.8 menemukan pergeseran sebaran dan kami membuat bobot untuk mengoreksinya. Memakai bobot itu sebagai sample weight di stacker adalah dugaan, bukan kepastian. AUC 0,59 tergolong lemah, dan bobotnya dipotong di 8, jadi bisa saja yang ditambahkan cuma noise.

Jadi kami uji langsung. Dua stacker dilatih pada fitur yang sama, satu tanpa bobot dan satu dengan bobot, lalu keduanya dinilai dengan metrik yang sama persis. Bobot ada di kedua sisi penilaian, jadi keunggulan yang muncul bukan berasal dari metriknya.

In [ ]:
Xa, Xb = rakit(HASIL[TERBAIK]["kelompok"])
tanpa, _, _ = stack_cv(Xa, None)
dengan, _, _ = stack_cv(Xa, bobot)

uji = pd.DataFrame({
    "dilatih tanpa bobot": {"MAE CV": mae(dari_log(tanpa)), "MAE ala-test": mae_test(dari_log(tanpa))},
    "dilatih dengan bobot": {"MAE CV": mae(dari_log(dengan)), "MAE ala-test": mae_test(dari_log(dengan))},
}).T.round(0)
uji["selisih ala-test"] = (uji["MAE ala-test"] - uji.loc["dilatih tanpa bobot", "MAE ala-test"]).round(0)
display(uji)

beda = uji.loc["dilatih tanpa bobot", "MAE ala-test"] - uji.loc["dilatih dengan bobot", "MAE ala-test"]
ambang = AMBANG
print(f"Selisih {beda:,.0f}. Ambang derau {ambang:,.0f}.")
print("Kesimpulan:", "bobot membantu di luar derau" if beda > ambang else
      ("bobot membantu tapi masih di dalam derau" if beda > 0 else "bobot tidak membantu pada run ini"))

Apa pun hasilnya, itulah yang kami laporkan. Kalau selisihnya di dalam derau, artinya bobot ini tidak boleh diklaim sebagai perbaikan, meskipun temuan pergeserannya sendiri tetap penting untuk membaca jarak CV dengan leaderboard.

## 6. Evaluasi dan analisis error

Semua di bagian ini memakai prediksi out-of-fold dari tahap terbaik.

### 6.1 Error menurut harga sebenarnya

In [ ]:
p = dari_log(HASIL[TERBAIK]["oof"])
rentang = pd.cut(y, [0, 1e5, 3e5, 6e5, 1e6, 2e6, 5e6, 1e10],
                 labels=["<100rb", "100-300rb", "300-600rb", "600rb-1jt", "1-2jt", "2-5jt", ">5jt"])
d = pd.DataFrame({"rentang": rentang, "err": np.abs(p - y), "y": y, "p": p})
g = d.groupby("rentang", observed=True)
tabel = pd.DataFrame({
    "jumlah": g.size(),
    "MAE": g["err"].mean(),
    "porsi_MAE": g["err"].sum() / d["err"].sum() * 100,
    "median p / median y": g["p"].median() / g["y"].median(),
})
display(tabel.round({"MAE": 0, "porsi_MAE": 1, "median p / median y": 2}))

fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
ax[0].hexbin(np.log10(y), np.log10(p), gridsize=50, cmap="viridis", mincnt=1, bins="log")
lim = [np.log10(y).min(), np.log10(y).max()]
ax[0].plot(lim, lim, color="red", lw=1)
ax[0].set_xlabel("log10(harga aktual)")
ax[0].set_ylabel("log10(prediksi)")
ax[0].set_title("Prediksi lawan aktual")
ax[1].bar(tabel.index.astype(str), tabel["porsi_MAE"], color="#c44e52")
ax[1].set_ylabel("% dari total error")
ax[1].set_title("Sumbangan tiap rentang harga terhadap error")
ax[1].tick_params(axis="x", rotation=25)
plt.tight_layout()
plt.show()

Kolom terakhir terlihat mencurigakan: rumah murah seperti ditebak terlalu mahal dan rumah sangat mahal ditebak terlalu murah. Pembacaan yang wajar adalah model menarik prediksi ke tengah, lalu menyimpulkan perlu koreksi khusus di segmen mahal.

Pembacaan itu salah, dan bagian berikutnya menjelaskan kenapa.

### 6.2 Kenapa tabel di atas tidak membuktikan adanya bias

Tabel 6.1 mengelompokkan baris berdasarkan `y`, yaitu berdasarkan hasil yang ingin ditebak. Mengelompokkan berdasarkan hasil selalu memunculkan pola menyusut ke tengah, bahkan pada model yang tidak bias sama sekali. Sebabnya, di dalam kelompok "harga di atas 5 juta" ikut terkumpul baris yang harganya tinggi karena faktor yang tidak terbaca dari teks, dan model memang tidak mungkin menebaknya.

Arah yang benar adalah mengelompokkan berdasarkan prediksi, lalu bertanya: dari semua listing yang kami tebak sekitar 1 juta, berapa median harga sebenarnya? Kalau jawabannya sekitar 1 juta, model terkalibrasi.

In [ ]:
desil = pd.qcut(p, 10, labels=False)
kal = pd.DataFrame({"d": desil, "p": p, "y": y}).groupby("d").median()
kal["rasio"] = kal["y"] / kal["p"]

fig, ax = plt.subplots(1, 2, figsize=(13, 4.2))
warna = ["#55a868" if 0.9 <= r <= 1.1 else "#c44e52" for r in kal["rasio"]]
ax[0].bar(range(10), kal["rasio"], color=warna)
ax[0].axhline(1.0, color="black", lw=1)
for i, r in enumerate(kal["rasio"]):
    ax[0].text(i, r + 0.02, f"{r:.2f}", ha="center", fontsize=8)
ax[0].set_ylim(0, max(1.35, kal["rasio"].max() * 1.15))
ax[0].set_xticks(range(10))
ax[0].set_xticklabels([f"D{i+1}" for i in range(10)])
ax[0].set_xlabel("desil prediksi")
ax[0].set_ylabel("median aktual / median prediksi")
ax[0].set_title("Dikelompokkan menurut PREDIKSI (cara yang benar)")

konv = tabel["median p / median y"]
ax[1].bar(range(len(konv)), konv, color="#4c72b0")
ax[1].axhline(1.0, color="black", lw=1)
ax[1].set_xticks(range(len(konv)))
ax[1].set_xticklabels(tabel.index.astype(str), rotation=25)
ax[1].set_ylabel("median prediksi / median aktual")
ax[1].set_title("Dikelompokkan menurut AKTUAL (menyesatkan)")
plt.tight_layout()
plt.show()

print("Rasio per desil prediksi:", np.round(kal["rasio"].values, 3))
print(f"Rentang: {kal['rasio'].min():.2f} sampai {kal['rasio'].max():.2f}")

Panel kiri jauh lebih datar daripada panel kanan. Kalau rasionya berada di sekitar 1,0, model sudah terkalibrasi dan tidak ada bias sistematis yang perlu dikoreksi.

Untuk memastikan, kami uji langsung: kalikan prediksi di desil teratas dengan beberapa faktor, lalu lihat MAE-nya.

In [ ]:
atas = p >= np.quantile(p, 0.9)
baris = []
for m in [0.9, 1.0, 1.05, 1.1, 1.25, 1.5]:
    q = p.copy()
    q[atas] = q[atas] * m
    baris.append({"pengali": m, "MAE CV": mae(q), "MAE ala-test": mae_test(q)})
uji_pengali = pd.DataFrame(baris).set_index("pengali").round(0)
display(uji_pengali)

terbaik_m = uji_pengali["MAE ala-test"].idxmin()
untung = uji_pengali.loc[1.0, "MAE ala-test"] - uji_pengali["MAE ala-test"].min()
print(f"Pengali terbaik: {terbaik_m}, keuntungan {untung:,.0f} dibanding tanpa koreksi.")
print(f"Ambang derau {AMBANG:,.0f}. Keuntungan itu {'di luar' if untung > AMBANG else 'di dalam'} derau.")

Hasilnya jelas. Menaikkan prediksi di segmen mahal tidak memberi keuntungan yang berarti, dan begitu pengalinya dinaikkan lebih jauh MAE langsung memburuk.

Ini masuk akal secara teori. MAE diminimalkan oleh median bersyarat. Ketika teks tidak memberi petunjuk cukup, median bersyarat memang mendekati median keseluruhan. Menyusut ke tengah bukan kelemahan yang perlu diperbaiki, melainkan perilaku yang benar untuk metrik ini.

Kesimpulan praktisnya: ekor mahal memang menguasai MAE, tetapi jalan keluarnya bukan mengoreksi skala prediksi. Yang dibutuhkan adalah sinyal baru yang benar-benar membedakan rumah 5 juta dari rumah 1 juta, dan sinyal itu harus datang dari teks.

## 7. File submission

In [ ]:
prediksi = dari_log(HASIL[TERBAIK]["tes"])
kirim = contoh[["id"]].merge(pd.DataFrame({"id": test["id"].values, "listPrice": np.round(prediksi, 2)}),
                             on="id", how="left")
kosong = int(kirim["listPrice"].isna().sum())
if kosong:
    print(f"peringatan: {kosong} id tanpa prediksi, diisi median (normal hanya kalau SUBSAMPLE aktif)")
    kirim["listPrice"] = kirim["listPrice"].fillna(float(np.median(prediksi)))
kirim.to_csv(f"{OUT}/submission.csv", index=False)

assert list(kirim.columns) == ["id", "listPrice"]
assert len(kirim) == len(contoh), "jumlah baris tidak sama dengan sample_submission"
assert (kirim["id"].values == contoh["id"].values).all(), "urutan id tidak sama"
assert kirim["listPrice"].notna().all() and (kirim["listPrice"] > 0).all()

print("submission.csv tersimpan")
print("baris:", len(kirim))
print("kuantil prediksi (1, 25, 50, 75, 99):", np.percentile(kirim["listPrice"], [1, 25, 50, 75, 99]).round(0))
print(f"median prediksi {kirim['listPrice'].median():,.0f} | median harga train {np.median(y):,.0f}")
lapor("selesai")
kirim.head()

In [ ]:
plt.figure(figsize=(7.5, 4))
b = np.linspace(4, 8, 60)
plt.hist(np.log10(np.clip(y, 1, None)), bins=b, density=True, alpha=0.65, label="harga train", color="#4c72b0")
plt.hist(np.log10(kirim["listPrice"]), bins=b, density=True, alpha=0.55, label="prediksi test", color="#c44e52")
plt.xlabel("log10(harga)")
plt.ylabel("kerapatan")
plt.title("Sebaran prediksi dibanding sebaran harga latih")
plt.legend()
plt.show()

Sebaran prediksi lebih rapat daripada sebaran aslinya, dan itu memang seharusnya begitu. Prediksi adalah median bersyarat, jadi variansnya lebih kecil daripada varians target. Kalau sebaran prediksi sama persis dengan sebaran harga latih, justru itu tanda model memaksakan diri menebak ekor dan akan dihukum oleh MAE.

## 8. Kesimpulan

Pendekatan kami adalah menggabungkan banyak representasi teks (TF-IDF, embedding kalimat, dan angka dari regex) dengan LightGBM yang dilatih pada log harga dengan loss L1 dan Huber. Angka tiap tahap ada di tabel bagian 5.3, dan urutan model tunggal ada di tabel bagian 4.3 serta keluaran tiap sel embedding.

Tiga hal yang kami anggap paling berharga dari pengerjaan ini:

Pertama, pemeriksaan sebaran train dan test di bagian 3.8. Tanpa itu kami akan terus mengoptimalkan angka yang salah. Jarak antara CV dan skor leaderboard jadi punya penjelasan, bukan sekadar dugaan bahwa subset publik kebetulan berat.

Kedua, koreksi cara membaca analisis error di bagian 6.2. Versi awal notebook ini menyimpulkan model bias ke tengah dan mengusulkan model khusus untuk listing mahal. Setelah diuji, arah itu justru merugikan. Kesalahan membaca seperti ini mudah terjadi dan mahal akibatnya kalau diikuti.

Ketiga, mengukur lantai derau lebih dulu di bagian 4.2, sehingga setiap selisih antar tahap bisa dinilai berarti atau tidak.

Hal yang perlu diingat saat membaca hasilnya:

- MAE didominasi rumah mahal, jadi selisih kecil antar tahap bisa masih di dalam derau. Angka ambangnya dicetak di bawah tabel 5.3.
- Skor leaderboard publik hanya memakai sebagian test. Submission kami sebelumnya dengan pipeline serupa mendapat skor publik sekitar 358 ribu, lebih tinggi daripada MAE CV, dan bagian 3.8 menjelaskan sebagian sebabnya.
- Teks adalah satu-satunya sumber informasi, jadi ada batas seberapa jauh harga bisa ditebak.
- Tidak ada data eksternal yang dipakai. Model yang dipakai adalah model pretrained publik, sesuai aturan lomba.

Yang belum sempat kami coba: fine-tuning penuh dengan lebih banyak epoch, karena di percobaan sebelumnya MAE validasi masih turun tiap epoch saat latihan kami hentikan, dan embedding yang lebih besar lagi. Keduanya mengarah ke hal yang sama, yaitu kualitas representasi teks, yang sejauh ini terbukti jadi pengungkit utama di soal ini.